# Kaniska FineCredit

An end-to-end, fairness-aware alternative-data credit-scoring project. The notebook uses the reproducible implementation in `src/` and the supplied synthetic dataset. It is designed to be run from the project root.

## Project objective

Estimate micro-loan repayment likelihood from consented mobile-money, utility, telco, location-stability, and rent-payment signals while exposing model performance and group-level fairness diagnostics. This is an educational reference implementation and not a production lending decision system.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'data' / 'credit_data.csv').exists() and (ROOT.parent / 'data' / 'credit_data.csv').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'credit_data.csv'
REPORTS_PATH = ROOT / 'reports'
print(f'Project root: {ROOT}')

In [ ]:
# Load the synthetic dataset and inspect its shape, schema, and target balance.
data = pd.read_csv(DATA_PATH)
print(f'Rows: {len(data):,}; columns: {data.shape[1]}')
display(data.head())
display(data[['gender', 'region', 'loan_repaid']].describe(include='all'))
display(data.isna().sum().sort_values(ascending=False).head(10).to_frame('missing_values'))

In [ ]:
# Engineer the derived behavioural features used by training and inference.
from src.feature_engineering import engineer_features, feature_columns

featured = engineer_features(data)
features = feature_columns(featured)
print(f'Engineered feature count: {len(features)}')
print(features)
display(featured[features].describe().T.head(15))

## Reproducible model pipeline

The training module compares Logistic Regression with optional XGBoost and LightGBM benchmarks, then fits a Fairlearn `ExponentiatedGradient` classifier constrained by demographic parity over intersectional gender-region groups. To retrain all artifacts, run the following cells or use the commands in `README.md`.

In [ ]:
# Uncomment to regenerate the dataset and retrain the model.
# from src.data_generation import main as generate_data
# from src.train import train
# generate_data()
# artifact = train()

model_path = ROOT / 'models' / 'fair_credit_model.pkl'
print(f'Model artifact available: {model_path.exists()} ({model_path})')

In [ ]:
# Read the measured evaluation output produced by src.evaluate.
evaluation = json.loads((REPORTS_PATH / 'evaluation.json').read_text(encoding='utf-8'))
summary = {key: evaluation[key] for key in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'demographic_parity_difference', 'equalized_odds_difference']}
display(pd.Series(summary, name='value').to_frame())

In [ ]:
# Inspect approval rates and fairness gaps by gender, region, and their intersection.
fairness = json.loads((REPORTS_PATH / 'fairness_report.json').read_text(encoding='utf-8'))
display(pd.DataFrame(fairness['approval_rates']))
gap_keys = [key for key in fairness if key.endswith('difference')]
display(pd.Series({key: fairness[key] for key in gap_keys}, name='gap').to_frame())

In [ ]:
# Optional single-applicant scoring example using the same API contract as the dashboard.
from src.api import Applicant, score

applicant = Applicant(
    age=35, gender='Female', region='Urban', mobile_money_txn_count_30d=24,
    mobile_money_avg_balance=250, mobile_money_inflow_std=30,
    mobile_money_outflow_std=30, mobile_money_unique_contacts=8,
    utility_on_time_payments_12m=9, utility_late_payments_12m=2,
    utility_avg_days_late=2, telco_airtime_topups_30d=8,
    telco_avg_topup_amount=25, telco_topup_trend=0.05,
    telco_location_stability=0.8, rent_payment_regularity=0.8,
)
if model_path.exists():
    result = score(applicant)
    display(pd.Series(result, name='applicant_result'))
else:
    print('Train the model before running the scoring example.')

## Interpretation and limitations

The reported model has an ROC-AUC of 0.6345 and an intersectional gender-region demographic-parity difference of 0.0540 in the supplied report. These values are diagnostic, not evidence of legal compliance or absence of discrimination. The dataset is synthetic; real deployment would require consent, privacy controls, monitoring for drift, documentation, adverse-action explanations, appeals, human review, and validation against applicable lending regulations.